In [1]:
!pip -q install rdkit torch-geometric tqdm scikit-learn requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.9 MB/s eta 0:00:00


In [2]:
import os
import time
import random
import requests
import numpy as np
import pandas as pd

from urllib.parse import quote
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score, average_precision_score

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

from torch_geometric.data import HeteroData
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import HANConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


In [10]:
# -----------------------------
# 1) Yeast gene → metabolite
# 학교 과제용 소형 curated dataset
# sterol / mevalonate / isoprenoid pathway 중심
# -----------------------------

gene_metabolite_edges = pd.DataFrame([
    # Early mevalonate pathway
    ["ERG10", "acetoacetyl-CoA"],
    ["ERG13", "3-hydroxy-3-methylglutaryl-CoA"],
    ["HMG1",  "mevalonic acid"],
    ["HMG2",  "mevalonic acid"],
    ["ERG12", "mevalonate 5-phosphate"],
    ["ERG8",  "mevalonate 5-diphosphate"],
    ["ERG19", "isopentenyl pyrophosphate"],

    # Isoprenoid intermediates
    ["IDI1",  "dimethylallyl pyrophosphate"],
    ["IDI1",  "isopentenyl pyrophosphate"],
    ["ERG20", "geranyl pyrophosphate"],
    ["ERG20", "farnesyl pyrophosphate"],
    ["BTS1",  "geranylgeranyl pyrophosphate"],

    # Sterol biosynthesis
    ["ERG9",  "squalene"],
    ["ERG1",  "2,3-oxidosqualene"],
    ["ERG7",  "lanosterol"],
    ["ERG11", "lanosterol"],
    ["ERG24", "zymosterol"],
    ["ERG6",  "fecosterol"],
    ["ERG2",  "episterol"],
    ["ERG3",  "ergosterol"],
    ["ERG5",  "ergosterol"],
    ["ERG4",  "ergosterol"],

    # Additional sterol-like intermediates / related compounds
    ["ERG11", "cholesterol"],
    ["ERG24", "desmosterol"],
    ["ERG6",  "campesterol"],
    ["ERG6",  "stigmasterol"],
    ["ERG5",  "beta-sitosterol"],

    # Small isoprenoid alcohols
    ["ERG20", "geraniol"],
    ["ERG20", "farnesol"],
    ["BTS1",  "geranylgeraniol"],
], columns=["gene_id", "metabolite_name"])


# -----------------------------
# 2) Human drug → target
# 탈모/androgen/sterol 관련 약물 + 비교용 약물
# -----------------------------

drug_target_edges = pd.DataFrame([
    # 5-alpha reductase inhibitors
    ["finasteride",      "SRD5A2", 0.95],
    ["finasteride",      "SRD5A1", 0.75],
    ["dutasteride",      "SRD5A1", 0.95],
    ["dutasteride",      "SRD5A2", 0.95],
    ["epristeride",      "SRD5A2", 0.80],

    # Androgen receptor antagonists
    ["spironolactone",   "AR",     0.70],
    ["bicalutamide",     "AR",     0.90],
    ["flutamide",        "AR",     0.85],
    ["nilutamide",       "AR",     0.85],
    ["enzalutamide",     "AR",     0.95],
    ["cyproterone acetate", "AR",  0.85],

    # Sterol / steroid metabolism related
    ["ketoconazole",     "CYP51A1",0.70],
    ["abiraterone",      "CYP17A1",0.90],
    ["clascoterone",     "AR",     0.80],

    # Hair-growth related but not androgen-axis direct inhibitor
    ["minoxidil",        "KCNJ8",  0.60],

    # Decoy / comparison drugs
    ["simvastatin",      "HMGCR",  0.80],
    ["atorvastatin",     "HMGCR",  0.80],
    ["tamoxifen",        "ESR1",   0.80],
], columns=["drug_name", "target_id", "confidence"])


# -----------------------------
# 3) Human target → disease
# Alopecia와 직접 관련성이 강한 축은 AR/SRD5A
# 나머지는 약한 연결 또는 비교용 연결로 둔다.
# -----------------------------

target_disease_edges = pd.DataFrame([
    ["SRD5A1",  "ALOPECIA", "Alopecia", 0.80],
    ["SRD5A2",  "ALOPECIA", "Alopecia", 1.00],
    ["AR",      "ALOPECIA", "Alopecia", 0.90],
    ["CYP17A1", "ALOPECIA", "Alopecia", 0.40],
    ["CYP51A1", "ALOPECIA", "Alopecia", 0.30],
    ["KCNJ8",   "ALOPECIA", "Alopecia", 0.40],

    # Decoy targets: graph에는 있지만 disease association은 약하게
    ["HMGCR",   "ALOPECIA", "Alopecia", 0.10],
    ["ESR1",    "ALOPECIA", "Alopecia", 0.10],
], columns=["target_id", "disease_id", "disease_name", "association_score"])


# -----------------------------
# 4) Node table
# -----------------------------

genes_df = pd.DataFrame({
    "gene_id": sorted(gene_metabolite_edges["gene_id"].unique())
})

metabolites_df = pd.DataFrame({
    "metabolite_name": sorted(gene_metabolite_edges["metabolite_name"].unique())
})

drugs_df = pd.DataFrame({
    "drug_name": sorted(drug_target_edges["drug_name"].unique())
})

targets_df = pd.DataFrame({
    "target_id": sorted(
        set(drug_target_edges["target_id"]) | set(target_disease_edges["target_id"])
    )
})

diseases_df = pd.DataFrame({
    "disease_id": ["ALOPECIA"],
    "disease_name": ["Alopecia"]
})

print("genes:", genes_df.shape)
print("metabolites:", metabolites_df.shape)
print("drugs:", drugs_df.shape)
print("targets:", targets_df.shape)
print("diseases:", diseases_df.shape)

display(gene_metabolite_edges.head(20))
display(drug_target_edges)
display(target_disease_edges)

genes: (20, 1)
metabolites: (25, 1)
drugs: (16, 1)
targets: (8, 1)
diseases: (1, 2)


,gene_id,metabolite_name
0,ERG10,acetoacetyl-CoA
1,ERG13,3-hydroxy-3-methylglutaryl-CoA
2,HMG1,mevalonic acid
3,HMG2,mevalonic acid
4,ERG12,mevalonate 5-phosphate
5,ERG8,mevalonate 5-diphosphate
6,ERG19,isopentenyl pyrophosphate
7,IDI1,dimethylallyl pyrophosphate
8,IDI1,isopentenyl pyrophosphate
9,ERG20,geranyl pyrophosphate


,drug_name,target_id,confidence
0,finasteride,SRD5A2,0.95
1,finasteride,SRD5A1,0.75
2,dutasteride,SRD5A1,0.95
3,dutasteride,SRD5A2,0.95
4,epristeride,SRD5A2,0.80
5,spironolactone,AR,0.70
6,bicalutamide,AR,0.90
7,flutamide,AR,0.85
8,nilutamide,AR,0.85
9,enzalutamide,AR,0.95


,target_id,disease_id,disease_name,association_score
0,SRD5A1,ALOPECIA,Alopecia,0.8
1,SRD5A2,ALOPECIA,Alopecia,1.0
2,AR,ALOPECIA,Alopecia,0.9
3,CYP17A1,ALOPECIA,Alopecia,0.4
4,CYP51A1,ALOPECIA,Alopecia,0.3
5,KCNJ8,ALOPECIA,Alopecia,0.4
6,HMGCR,ALOPECIA,Alopecia,0.1
7,ESR1,ALOPECIA,Alopecia,0.1


In [11]:
def fetch_canonical_smiles_from_pubchem(compound_name, sleep_sec=0.2):
    """
    PubChem PUG-REST에서 compound name으로 CanonicalSMILES를 가져온다.
    실패하면 None 반환.
    """
    safe_name = quote(compound_name)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{safe_name}/property/CanonicalSMILES/JSON"

    try:
        r = requests.get(url, timeout=20)
        time.sleep(sleep_sec)

        if r.status_code != 200:
            print(f"[WARN] PubChem failed: {compound_name}, status={r.status_code}")
            return None

        data = r.json()
        props = data["PropertyTable"]["Properties"]
        if len(props) == 0:
            return None

        return props[0].get("ConnectivitySMILES", None)

    except Exception as e:
        print(f"[WARN] PubChem error: {compound_name}, {e}")
        return None

In [12]:
# metabolite SMILES
metabolites_df["smiles"] = [
    fetch_canonical_smiles_from_pubchem(name)
    for name in tqdm(metabolites_df["metabolite_name"], desc="Fetching metabolite SMILES")
]

# drug SMILES
drugs_df["smiles"] = [
    fetch_canonical_smiles_from_pubchem(name)
    for name in tqdm(drugs_df["drug_name"], desc="Fetching drug SMILES")
]

display(metabolites_df)
display(drugs_df)

Fetching metabolite SMILES:   0%|          | 0/25 [00:00<?, ?it/s]

Fetching drug SMILES:   0%|          | 0/16 [00:00<?, ?it/s]

,metabolite_name,smiles
0,"2,3-oxidosqualene",CC(=CCCC(=CCCC(=CCCC=C(C)CCC=C(C)CCC1C(O1)(C)C...
1,3-hydroxy-3-methylglutaryl-CoA,CC(C)(COP(=O)(O)OP(=O)(O)OCC1C(C(C(O1)N2C=NC3=...
2,acetoacetyl-CoA,CC(=O)CC(=O)SCCNC(=O)CCNC(=O)C(C(C)(C)COP(=O)(...
3,beta-sitosterol,CCC(CCC(C)C1CCC2C1(CCC3C2CC=C4C3(CCC(C4)O)C)C)...
4,campesterol,CC(C)C(C)CCC(C)C1CCC2C1(CCC3C2CC=C4C3(CCC(C4)O...
5,cholesterol,CC(C)CCCC(C)C1CCC2C1(CCC3C2CC=C4C3(CCC(C4)O)C)C
6,desmosterol,CC(CCC=C(C)C)C1CCC2C1(CCC3C2CC=C4C3(CCC(C4)O)C)C
7,dimethylallyl pyrophosphate,CC(=CCOP(=O)(O)OP(=O)(O)O)C
8,episterol,CC(C)C(=C)CCC(C)C1CCC2C1(CCC3C2=CCC4C3(CCC(C4)...
9,ergosterol,CC(C)C(C)C=CC(C)C1CCC2C1(CCC3C2=CC=C4C3(CCC(C4...


,drug_name,smiles
0,abiraterone,CC12CCC(CC1=CCC3C2CCC4(C3CC=C4C5=CN=CC=C5)C)O
1,atorvastatin,CC(C)C1=C(C(=C(N1CCC(CC(CC(=O)O)O)O)C2=CC=C(C=...
2,bicalutamide,CC(CS(=O)(=O)C1=CC=C(C=C1)F)(C(=O)NC2=CC(=C(C=...
3,clascoterone,CCC(=O)OC1(CCC2C1(CCC3C2CCC4=CC(=O)CCC34C)C)C(...
4,cyproterone acetate,CC(=O)C1(CCC2C1(CCC3C2C=C(C4=CC(=O)C5CC5C34C)C...
5,dutasteride,CC12CCC3C(C1CCC2C(=O)NC4=C(C=CC(=C4)C(F)(F)F)C...
6,enzalutamide,CC1(C(=O)N(C(=S)N1C2=CC(=C(C=C2)C(=O)NC)F)C3=C...
7,epristeride,CC12CCC3C(C1CCC2C(=O)NC(C)(C)C)CC=C4C3(CCC(=C4...
8,finasteride,CC12CCC3C(C1CCC2C(=O)NC(C)(C)C)CCC4C3(C=CC(=O)...
9,flutamide,CC(C)C(=O)NC1=CC(=C(C=C1)[N+](=O)[O-])C(F)(F)F


In [13]:
metabolites_df = metabolites_df.dropna(subset=["smiles"]).reset_index(drop=True)
drugs_df = drugs_df.dropna(subset=["smiles"]).reset_index(drop=True)

# gene-metabolite edge도 살아남은 metabolite만 유지
gene_metabolite_edges = gene_metabolite_edges[
    gene_metabolite_edges["metabolite_name"].isin(metabolites_df["metabolite_name"])
].reset_index(drop=True)

# drug-target edge도 살아남은 drug만 유지
drug_target_edges = drug_target_edges[
    drug_target_edges["drug_name"].isin(drugs_df["drug_name"])
].reset_index(drop=True)

print("After SMILES filtering")
print("metabolites:", metabolites_df.shape)
print("drugs:", drugs_df.shape)
print("gene-metabolite edges:", gene_metabolite_edges.shape)
print("drug-target edges:", drug_target_edges.shape)

After SMILES filtering
metabolites: (25, 2)
drugs: (16, 2)
gene-metabolite edges: (30, 2)
drug-target edges: (18, 3)


In [14]:
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def smiles_to_mol(smiles):
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    return mol

def smiles_to_fp(smiles):
    mol = smiles_to_mol(smiles)
    if mol is None:
        return None
    return morgan_generator.GetFingerprint(mol)

metabolites_df["mol"] = metabolites_df["smiles"].apply(smiles_to_mol)
metabolites_df["fp"] = metabolites_df["smiles"].apply(smiles_to_fp)

drugs_df["mol"] = drugs_df["smiles"].apply(smiles_to_mol)
drugs_df["fp"] = drugs_df["smiles"].apply(smiles_to_fp)

metabolites_df = metabolites_df[metabolites_df["fp"].notna()].reset_index(drop=True)
drugs_df = drugs_df[drugs_df["fp"].notna()].reset_index(drop=True)

print("valid metabolites:", len(metabolites_df))
print("valid drugs:", len(drugs_df))

valid metabolites: 25
valid drugs: 16


In [16]:
TOP_K = 5

sim_rows = []

drug_names = drugs_df["drug_name"].tolist()
drug_fps = drugs_df["fp"].tolist()

for met_row in tqdm(metabolites_df.itertuples(index=False), total=len(metabolites_df)):
    met_name = met_row.metabolite_name
    met_fp = met_row.fp

    sims = DataStructs.BulkTanimotoSimilarity(met_fp, drug_fps)
    top_idx = np.argsort(sims)[::-1][:TOP_K]

    for j in top_idx:
        sim_rows.append({
            "metabolite_name": met_name,
            "drug_name": drug_names[j],
            "tanimoto": float(sims[j])
        })

metabolite_drug_edges = pd.DataFrame(sim_rows)

print("Metabolite-Drug similarity edges:", metabolite_drug_edges.shape)

display(
    metabolite_drug_edges
    .sort_values("tanimoto", ascending=False)
    .head(50)
)

  0%|          | 0/25 [00:00<?, ?it/s]

Metabolite-Drug similarity edges: (125, 3)


,metabolite_name,drug_name,tanimoto
15,beta-sitosterol,abiraterone,0.434783
20,campesterol,abiraterone,0.432836
25,cholesterol,abiraterone,0.432836
30,desmosterol,abiraterone,0.420290
115,stigmasterol,abiraterone,0.408451
26,cholesterol,epristeride,0.294872
21,campesterol,epristeride,0.294872
31,desmosterol,epristeride,0.287500
16,beta-sitosterol,epristeride,0.283951
116,stigmasterol,epristeride,0.280488


In [17]:
def make_mapping(values):
    values = sorted(pd.Series(values).dropna().astype(str).unique().tolist())
    id2idx = {v: i for i, v in enumerate(values)}
    idx2id = {i: v for v, i in id2idx.items()}
    return id2idx, idx2id

gene2idx, idx2gene = make_mapping(genes_df["gene_id"])
met2idx, idx2met = make_mapping(metabolites_df["metabolite_name"])
drug2idx, idx2drug = make_mapping(drugs_df["drug_name"])
target2idx, idx2target = make_mapping(targets_df["target_id"])
disease2idx, idx2disease = make_mapping(diseases_df["disease_id"])

print("gene:", len(gene2idx))
print("metabolite:", len(met2idx))
print("drug:", len(drug2idx))
print("target:", len(target2idx))
print("disease:", len(disease2idx))

gene: 20
metabolite: 25
drug: 16
target: 8
disease: 1


In [18]:
def build_edge_index(df, src_col, dst_col, src_map, dst_map):
    src_list = []
    dst_list = []

    for _, row in df.iterrows():
        s = str(row[src_col])
        d = str(row[dst_col])

        if s in src_map and d in dst_map:
            src_list.append(src_map[s])
            dst_list.append(dst_map[d])

    if len(src_list) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    return torch.tensor([src_list, dst_list], dtype=torch.long)


data = HeteroData()

data["gene"].num_nodes = len(gene2idx)
data["metabolite"].num_nodes = len(met2idx)
data["drug"].num_nodes = len(drug2idx)
data["target"].num_nodes = len(target2idx)
data["disease"].num_nodes = len(disease2idx)


# Gene -> Metabolite
gm_edge = build_edge_index(
    gene_metabolite_edges,
    "gene_id",
    "metabolite_name",
    gene2idx,
    met2idx
)

data["gene", "produces", "metabolite"].edge_index = gm_edge
data["metabolite", "rev_produces", "gene"].edge_index = gm_edge.flip(0)


# Metabolite -> Drug
md_edge = build_edge_index(
    metabolite_drug_edges,
    "metabolite_name",
    "drug_name",
    met2idx,
    drug2idx
)

data["metabolite", "similar_to", "drug"].edge_index = md_edge
data["drug", "rev_similar_to", "metabolite"].edge_index = md_edge.flip(0)


# Drug -> Target
dt_edge = build_edge_index(
    drug_target_edges,
    "drug_name",
    "target_id",
    drug2idx,
    target2idx
)

data["drug", "inhibits", "target"].edge_index = dt_edge
data["target", "rev_inhibits", "drug"].edge_index = dt_edge.flip(0)


# Target -> Disease
td_edge = build_edge_index(
    target_disease_edges,
    "target_id",
    "disease_id",
    target2idx,
    disease2idx
)

data["target", "associated_with", "disease"].edge_index = td_edge
data["disease", "rev_associated_with", "target"].edge_index = td_edge.flip(0)


print(data)
print(data.metadata())

HeteroData(
  gene={ num_nodes=20 },
  metabolite={ num_nodes=25 },
  drug={ num_nodes=16 },
  target={ num_nodes=8 },
  disease={ num_nodes=1 },
  (gene, produces, metabolite)={ edge_index=[2, 30] },
  (metabolite, rev_produces, gene)={ edge_index=[2, 30] },
  (metabolite, similar_to, drug)={ edge_index=[2, 125] },
  (drug, rev_similar_to, metabolite)={ edge_index=[2, 125] },
  (drug, inhibits, target)={ edge_index=[2, 18] },
  (target, rev_inhibits, drug)={ edge_index=[2, 18] },
  (target, associated_with, disease)={ edge_index=[2, 8] },
  (disease, rev_associated_with, target)={ edge_index=[2, 8] }
)
(['gene', 'metabolite', 'drug', 'target', 'disease'], [('gene', 'produces', 'metabolite'), ('metabolite', 'rev_produces', 'gene'), ('metabolite', 'similar_to', 'drug'), ('drug', 'rev_similar_to', 'metabolite'), ('drug', 'inhibits', 'target'), ('target', 'rev_inhibits', 'drug'), ('target', 'associated_with', 'disease'), ('disease', 'rev_associated_with', 'target')])


In [19]:
TASK_EDGE_TYPES = [
    ("metabolite", "similar_to", "drug"),
    ("drug", "inhibits", "target"),
]

REV_EDGE_TYPES = [
    ("drug", "rev_similar_to", "metabolite"),
    ("target", "rev_inhibits", "drug"),
]

transform = RandomLinkSplit(
    num_val=0.2,
    num_test=0.2,
    is_undirected=False,
    add_negative_train_samples=True,
    neg_sampling_ratio=1.0,
    edge_types=TASK_EDGE_TYPES,
    rev_edge_types=REV_EDGE_TYPES,
)

train_data, val_data, test_data = transform(data)

print(train_data)

HeteroData(
  gene={ num_nodes=20 },
  metabolite={ num_nodes=25 },
  drug={ num_nodes=16 },
  target={ num_nodes=8 },
  disease={ num_nodes=1 },
  (gene, produces, metabolite)={ edge_index=[2, 30] },
  (metabolite, rev_produces, gene)={ edge_index=[2, 30] },
  (metabolite, similar_to, drug)={
    edge_index=[2, 75],
    edge_label=[150],
    edge_label_index=[2, 150],
  },
  (drug, rev_similar_to, metabolite)={ edge_index=[2, 75] },
  (drug, inhibits, target)={
    edge_index=[2, 12],
    edge_label=[24],
    edge_label_index=[2, 24],
  },
  (target, rev_inhibits, drug)={ edge_index=[2, 12] },
  (target, associated_with, disease)={ edge_index=[2, 8] },
  (disease, rev_associated_with, target)={ edge_index=[2, 8] }
)


In [20]:
class SimpleHANLinkPredictor(nn.Module):
    def __init__(
        self,
        metadata,
        num_nodes_dict,
        hidden_channels=64,
        out_channels=64,
        heads=4,
        dropout=0.1,
    ):
        super().__init__()

        self.node_emb = nn.ModuleDict({
            node_type: nn.Embedding(num_nodes, hidden_channels)
            for node_type, num_nodes in num_nodes_dict.items()
        })

        for emb in self.node_emb.values():
            nn.init.normal_(emb.weight, mean=0.0, std=0.1)

        self.conv1 = HANConv(
            in_channels=hidden_channels,
            out_channels=hidden_channels,
            metadata=metadata,
            heads=heads,
            dropout=dropout,
        )

        self.conv2 = HANConv(
            in_channels=hidden_channels,
            out_channels=out_channels,
            metadata=metadata,
            heads=heads,
            dropout=dropout,
        )

    def forward(self, data):
        x_dict = {
            node_type: emb.weight
            for node_type, emb in self.node_emb.items()
        }

        out1 = self.conv1(x_dict, data.edge_index_dict)
        x_dict = {
            k: F.elu(out1[k]) if out1.get(k) is not None else x_dict[k]
            for k in x_dict.keys()
        }

        out2 = self.conv2(x_dict, data.edge_index_dict)
        x_dict = {
            k: F.elu(out2[k]) if out2.get(k) is not None else x_dict[k]
            for k in x_dict.keys()
        }

        return x_dict

    def decode(self, z_dict, edge_label_index, src_type, dst_type):
        src_z = z_dict[src_type][edge_label_index[0]]
        dst_z = z_dict[dst_type][edge_label_index[1]]
        logits = (src_z * dst_z).sum(dim=-1)
        return logits

In [21]:
num_nodes_dict = {
    "gene": data["gene"].num_nodes,
    "metabolite": data["metabolite"].num_nodes,
    "drug": data["drug"].num_nodes,
    "target": data["target"].num_nodes,
    "disease": data["disease"].num_nodes,
}

model = SimpleHANLinkPredictor(
    metadata=data.metadata(),
    num_nodes_dict=num_nodes_dict,
    hidden_channels=64,
    out_channels=64,
    heads=4,
    dropout=0.1,
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=1e-4,
)

train_data = train_data.to(DEVICE)
val_data = val_data.to(DEVICE)
test_data = test_data.to(DEVICE)

print(model)

SimpleHANLinkPredictor(
  (node_emb): ModuleDict(
    (gene): Embedding(20, 64)
    (metabolite): Embedding(25, 64)
    (drug): Embedding(16, 64)
    (target): Embedding(8, 64)
    (disease): Embedding(1, 64)
  )
  (conv1): HANConv(64, heads=4)
  (conv2): HANConv(64, heads=4)
)


In [22]:
def compute_loss(model, batch_data, task_edge_types):
    z_dict = model(batch_data)
    losses = []

    for edge_type in task_edge_types:
        src_type, rel_type, dst_type = edge_type

        edge_label_index = batch_data[edge_type].edge_label_index
        edge_label = batch_data[edge_type].edge_label.float()

        logits = model.decode(
            z_dict,
            edge_label_index,
            src_type,
            dst_type,
        )

        loss = F.binary_cross_entropy_with_logits(logits, edge_label)
        losses.append(loss)

    return sum(losses) / len(losses)


@torch.no_grad()
def evaluate_auc_ap(model, batch_data, task_edge_types):
    model.eval()
    z_dict = model(batch_data)

    results = {}

    for edge_type in task_edge_types:
        src_type, rel_type, dst_type = edge_type

        edge_label_index = batch_data[edge_type].edge_label_index
        edge_label = batch_data[edge_type].edge_label.float()

        logits = model.decode(
            z_dict,
            edge_label_index,
            src_type,
            dst_type,
        )

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        y_true = edge_label.detach().cpu().numpy()

        if len(np.unique(y_true)) < 2:
            auc = np.nan
            ap = np.nan
        else:
            auc = roc_auc_score(y_true, probs)
            ap = average_precision_score(y_true, probs)

        results[edge_type] = {
            "AUC": auc,
            "AP": ap,
        }

    return results

In [23]:
EPOCHS = 100

best_val_ap = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    loss = compute_loss(model, train_data, TASK_EDGE_TYPES)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        val_result = evaluate_auc_ap(model, val_data, TASK_EDGE_TYPES)
        mean_val_ap = np.nanmean([v["AP"] for v in val_result.values()])

        if mean_val_ap > best_val_ap:
            best_val_ap = mean_val_ap
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(f"Epoch {epoch:03d} | Loss {loss.item():.4f} | Val mean AP {mean_val_ap:.4f}")
        for etype, metrics in val_result.items():
            print(" ", etype, metrics)

if best_state is not None:
    model.load_state_dict(best_state)

print("Training finished.")

Epoch 010 | Loss 0.6093 | Val mean AP 0.5986
  ('metabolite', 'similar_to', 'drug') {'AUC': np.float64(0.68), 'AP': np.float64(0.697287145704527)}
  ('drug', 'inhibits', 'target') {'AUC': np.float64(0.2777777777777778), 'AP': np.float64(0.5)}
Epoch 020 | Loss 0.5305 | Val mean AP 0.6600
  ('metabolite', 'similar_to', 'drug') {'AUC': np.float64(0.7096), 'AP': np.float64(0.7088353925409954)}
  ('drug', 'inhibits', 'target') {'AUC': np.float64(0.5), 'AP': np.float64(0.6111111111111112)}
Epoch 030 | Loss 0.4779 | Val mean AP 0.6345
  ('metabolite', 'similar_to', 'drug') {'AUC': np.float64(0.6672), 'AP': np.float64(0.657973226890702)}
  ('drug', 'inhibits', 'target') {'AUC': np.float64(0.5555555555555556), 'AP': np.float64(0.6111111111111112)}
Epoch 040 | Loss 0.4539 | Val mean AP 0.5743
  ('metabolite', 'similar_to', 'drug') {'AUC': np.float64(0.692), 'AP': np.float64(0.6485063216079773)}
  ('drug', 'inhibits', 'target') {'AUC': np.float64(0.16666666666666669), 'AP': np.float64(0.5)}
Epoch

In [24]:
@torch.no_grad()
def recall_at_k(model, batch_data, edge_type, k=3):
    model.eval()
    z_dict = model(batch_data)

    src_type, rel_type, dst_type = edge_type

    edge_label_index = batch_data[edge_type].edge_label_index
    edge_label = batch_data[edge_type].edge_label

    pos_edge_index = edge_label_index[:, edge_label == 1]

    if pos_edge_index.size(1) == 0:
        return np.nan

    dst_all = z_dict[dst_type]
    hits = 0

    for i in range(pos_edge_index.size(1)):
        src_idx = pos_edge_index[0, i]
        true_dst_idx = pos_edge_index[1, i]

        src_vec = z_dict[src_type][src_idx]
        scores = torch.matmul(dst_all, src_vec)

        topk = torch.topk(scores, k=min(k, dst_all.size(0))).indices

        if (topk == true_dst_idx).any():
            hits += 1

    return hits / pos_edge_index.size(1)


for k in [1, 2, 3]:
    r1 = recall_at_k(
        model,
        test_data,
        edge_type=("drug", "inhibits", "target"),
        k=k,
    )

    r2 = recall_at_k(
        model,
        test_data,
        edge_type=("metabolite", "similar_to", "drug"),
        k=k,
    )

    print(f"Drug-Target Recall@{k}: {r1:.4f}")
    print(f"Metabolite-Drug Recall@{k}: {r2:.4f}")

Drug-Target Recall@1: 0.3333
Metabolite-Drug Recall@1: 0.1600
Drug-Target Recall@2: 0.3333
Metabolite-Drug Recall@2: 0.2400
Drug-Target Recall@3: 0.6667
Metabolite-Drug Recall@3: 0.2800


In [25]:
@torch.no_grad()
def rank_genes_for_disease(model, full_data, disease_id="ALOPECIA", top_k=20):
    model.eval()

    full_data = full_data.to(DEVICE)
    z_dict = model(full_data)

    disease_idx = disease2idx[disease_id]
    disease_vec = z_dict["disease"][disease_idx]
    gene_mat = z_dict["gene"]

    logits = torch.matmul(gene_mat, disease_vec)
    probs = torch.sigmoid(logits)

    top = torch.topk(probs, k=min(top_k, probs.size(0)))

    rows = []
    for rank, (idx, score) in enumerate(
        zip(top.indices.cpu().tolist(), top.values.cpu().tolist()),
        start=1
    ):
        rows.append({
            "rank": rank,
            "gene_id": idx2gene[idx],
            "gnn_score": float(score)
        })

    return pd.DataFrame(rows)


gene_rank_df = rank_genes_for_disease(
    model,
    data,
    disease_id="ALOPECIA",
    top_k=20,
)

display(gene_rank_df)

,rank,gene_id,gnn_score
0,1,ERG10,0.999065
1,2,ERG12,0.994753
2,3,HMG2,0.948491
3,4,HMG1,0.948491
4,5,ERG5,0.573148
5,6,ERG4,0.561893
6,7,ERG3,0.561893
7,8,ERG19,0.504427
8,9,ERG8,0.501808
9,10,ERG2,0.501793


In [26]:
def build_path_explanation():
    rows = []

    for _, gm in gene_metabolite_edges.iterrows():
        gene = gm["gene_id"]
        met = gm["metabolite_name"]

        sim_hits = metabolite_drug_edges[
            metabolite_drug_edges["metabolite_name"] == met
        ]

        for _, sim in sim_hits.iterrows():
            drug = sim["drug_name"]
            tanimoto = sim["tanimoto"]

            dt_hits = drug_target_edges[
                drug_target_edges["drug_name"] == drug
            ]

            for _, dt in dt_hits.iterrows():
                target = dt["target_id"]
                dt_conf = dt["confidence"]

                td_hits = target_disease_edges[
                    target_disease_edges["target_id"] == target
                ]

                for _, td in td_hits.iterrows():
                    disease = td["disease_id"]
                    disease_score = td["association_score"]

                    path_score = tanimoto * dt_conf * disease_score

                    rows.append({
                        "gene_id": gene,
                        "metabolite": met,
                        "similar_drug": drug,
                        "target": target,
                        "disease": disease,
                        "tanimoto": tanimoto,
                        "drug_target_confidence": dt_conf,
                        "target_disease_score": disease_score,
                        "path_score": path_score,
                    })

    return pd.DataFrame(rows)


path_df = build_path_explanation()

display(
    path_df.sort_values("path_score", ascending=False).head(30)
)

,gene_id,metabolite,similar_drug,target,disease,tanimoto,drug_target_confidence,target_disease_score,path_score
132,ERG6,campesterol,epristeride,SRD5A2,ALOPECIA,0.294872,0.80,1.0,0.235897
120,ERG11,cholesterol,epristeride,SRD5A2,ALOPECIA,0.294872,0.80,1.0,0.235897
126,ERG24,desmosterol,epristeride,SRD5A2,ALOPECIA,0.287500,0.80,1.0,0.230000
144,ERG5,beta-sitosterol,epristeride,SRD5A2,ALOPECIA,0.283951,0.80,1.0,0.227160
138,ERG6,stigmasterol,epristeride,SRD5A2,ALOPECIA,0.280488,0.80,1.0,0.224390
122,ERG11,cholesterol,finasteride,SRD5A2,ALOPECIA,0.207317,0.95,1.0,0.196951
134,ERG6,campesterol,finasteride,SRD5A2,ALOPECIA,0.207317,0.95,1.0,0.196951
129,ERG24,desmosterol,finasteride,SRD5A2,ALOPECIA,0.202381,0.95,1.0,0.192262
146,ERG5,beta-sitosterol,finasteride,SRD5A2,ALOPECIA,0.200000,0.95,1.0,0.190000
140,ERG6,stigmasterol,finasteride,SRD5A2,ALOPECIA,0.197674,0.95,1.0,0.187791


In [27]:
# gene별 최고 path score
gene_path_score = (
    path_df
    .groupby("gene_id")["path_score"]
    .max()
    .reset_index()
)

# 0~1 scaling
if len(gene_path_score) > 0:
    max_path = gene_path_score["path_score"].max()
    if max_path > 0:
        gene_path_score["path_score_scaled"] = gene_path_score["path_score"] / max_path
    else:
        gene_path_score["path_score_scaled"] = 0.0
else:
    gene_path_score["path_score_scaled"] = 0.0

final_rank_df = gene_rank_df.merge(
    gene_path_score[["gene_id", "path_score", "path_score_scaled"]],
    on="gene_id",
    how="left"
)

final_rank_df["path_score"] = final_rank_df["path_score"].fillna(0)
final_rank_df["path_score_scaled"] = final_rank_df["path_score_scaled"].fillna(0)

final_rank_df["final_score"] = (
    0.5 * final_rank_df["gnn_score"]
    + 0.5 * final_rank_df["path_score_scaled"]
)

final_rank_df = final_rank_df.sort_values(
    "final_score",
    ascending=False
).reset_index(drop=True)

final_rank_df["final_rank"] = np.arange(1, len(final_rank_df) + 1)

display(final_rank_df[
    ["final_rank", "gene_id", "gnn_score", "path_score", "path_score_scaled", "final_score"]
])

,final_rank,gene_id,gnn_score,path_score,path_score_scaled,final_score
0,1,ERG5,0.573148,0.227160,0.962963,0.768055
1,2,ERG6,0.500000,0.235897,1.000000,0.750000
2,3,ERG11,0.500000,0.235897,1.000000,0.750000
3,4,ERG24,0.500000,0.230000,0.975000,0.737500
4,5,ERG10,0.999065,0.098016,0.415502,0.707283
5,6,ERG12,0.994753,0.095000,0.402717,0.698735
6,7,HMG1,0.948491,0.104516,0.443058,0.695774
7,8,HMG2,0.948491,0.104516,0.443058,0.695774
8,9,ERG3,0.561893,0.132558,0.561931,0.561912
9,10,ERG4,0.561893,0.132558,0.561931,0.561912


In [28]:
TOP_N = 5

for gene in final_rank_df["gene_id"].head(TOP_N):
    print("=" * 80)
    print(f"Gene: {gene}")

    gene_paths = path_df[path_df["gene_id"] == gene].sort_values(
        "path_score",
        ascending=False
    )

    if len(gene_paths) == 0:
        print("No interpretable path found.")
    else:
        display(gene_paths.head(5))

Gene: ERG5


,gene_id,metabolite,similar_drug,target,disease,tanimoto,drug_target_confidence,target_disease_score,path_score
144,ERG5,beta-sitosterol,epristeride,SRD5A2,ALOPECIA,0.283951,0.80,1.0,0.227160
146,ERG5,beta-sitosterol,finasteride,SRD5A2,ALOPECIA,0.200000,0.95,1.0,0.190000
149,ERG5,beta-sitosterol,dutasteride,SRD5A2,ALOPECIA,0.185567,0.95,1.0,0.176289
145,ERG5,beta-sitosterol,clascoterone,AR,ALOPECIA,0.223529,0.80,0.9,0.160941
143,ERG5,beta-sitosterol,abiraterone,CYP17A1,ALOPECIA,0.434783,0.90,0.4,0.156522


Gene: ERG6


,gene_id,metabolite,similar_drug,target,disease,tanimoto,drug_target_confidence,target_disease_score,path_score
132,ERG6,campesterol,epristeride,SRD5A2,ALOPECIA,0.294872,0.80,1.0,0.235897
138,ERG6,stigmasterol,epristeride,SRD5A2,ALOPECIA,0.280488,0.80,1.0,0.224390
134,ERG6,campesterol,finasteride,SRD5A2,ALOPECIA,0.207317,0.95,1.0,0.196951
140,ERG6,stigmasterol,finasteride,SRD5A2,ALOPECIA,0.197674,0.95,1.0,0.187791
139,ERG6,stigmasterol,clascoterone,AR,ALOPECIA,0.235294,0.80,0.9,0.169412


Gene: ERG11


,gene_id,metabolite,similar_drug,target,disease,tanimoto,drug_target_confidence,target_disease_score,path_score
120,ERG11,cholesterol,epristeride,SRD5A2,ALOPECIA,0.294872,0.80,1.0,0.235897
122,ERG11,cholesterol,finasteride,SRD5A2,ALOPECIA,0.207317,0.95,1.0,0.196951
121,ERG11,cholesterol,clascoterone,AR,ALOPECIA,0.216867,0.80,0.9,0.156145
119,ERG11,cholesterol,abiraterone,CYP17A1,ALOPECIA,0.432836,0.90,0.4,0.155821
123,ERG11,cholesterol,finasteride,SRD5A1,ALOPECIA,0.207317,0.75,0.8,0.124390


Gene: ERG24


,gene_id,metabolite,similar_drug,target,disease,tanimoto,drug_target_confidence,target_disease_score,path_score
126,ERG24,desmosterol,epristeride,SRD5A2,ALOPECIA,0.287500,0.80,1.0,0.230000
129,ERG24,desmosterol,finasteride,SRD5A2,ALOPECIA,0.202381,0.95,1.0,0.192262
127,ERG24,desmosterol,clascoterone,AR,ALOPECIA,0.226190,0.80,0.9,0.162857
125,ERG24,desmosterol,abiraterone,CYP17A1,ALOPECIA,0.420290,0.90,0.4,0.151304
128,ERG24,desmosterol,spironolactone,AR,ALOPECIA,0.211765,0.70,0.9,0.133412


Gene: ERG10


,gene_id,metabolite,similar_drug,target,disease,tanimoto,drug_target_confidence,target_disease_score,path_score
3,ERG10,acetoacetyl-CoA,finasteride,SRD5A2,ALOPECIA,0.103175,0.95,1.0,0.098016
2,ERG10,acetoacetyl-CoA,enzalutamide,AR,ALOPECIA,0.106061,0.95,0.9,0.090682
5,ERG10,acetoacetyl-CoA,flutamide,AR,ALOPECIA,0.095652,0.85,0.9,0.073174
4,ERG10,acetoacetyl-CoA,finasteride,SRD5A1,ALOPECIA,0.103175,0.75,0.8,0.061905
0,ERG10,acetoacetyl-CoA,ketoconazole,CYP51A1,ALOPECIA,0.129496,0.70,0.3,0.027194
